# Detecting Malicious URLs with Machine Learning

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install -q scikit-learn pandas matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import re
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))
df.head()
def extract_features(df):
    df['url_length'] = df['url'].apply(len)
    df['num_dots'] = df['url'].apply(lambda x: x.count('.'))
    df['num_slashes'] = df['url'].apply(lambda x: x.count('/'))
    df['has_ip'] = df['url'].apply(lambda x: 1 if re.search(r'(\d+\.\d+\.\d+\.\d+)', x) else 0)
    return df

df = extract_features(df)
df.head()
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
X_tfidf = tfidf.fit_transform(df['url'])
X = np.hstack((X_tfidf.toarray(), df[['url_length', 'num_dots', 'num_slashes', 'has_ip']].values))
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_pred))
